# 1) Forecasting basics: baselines, metrics and honest evaluation

## Objectives

At the end of this notebook you should be able to:

- split energy time series for forecasting **in time**, never at random
- state exactly what is known when a **day-ahead forecast** is made — and spot forecasts that use data from the future
- build the three **baselines** every forecast must beat: persistence, seasonal naive and climatology
- measure accuracy with **MAE, RMSE, bias, MAPE and MASE**, and know what each one hides
- analyse **where** a forecast fails — by hour, by day type, on holidays

Every dispatch model in Module 03 took demand as known. In reality the day-ahead market clears on **forecasts**: a TSO, a supplier or a trader must predict tomorrow's demand by noon today. This module is about making those forecasts — and, first, about judging them honestly. Most forecasting mistakes are not bad models but unfair comparisons.

The lessons use **synthetic, Croatia-sized data** (2023–2025) from `forecast_data.py` in this folder: demand driven by temperature, with heating, cooling, tourism, a daily and weekly rhythm, Croatian public holidays and weather that persists for days. Because we know how the data were made, we can check what each method gets right. A real-data cell at the end compares with the forecast Croatia's TSO publishes on ENTSO-E.

In [1]:
import logging
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from forecast_data import load_data

warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("entsoe").setLevel(logging.ERROR)

### The data

In [2]:
df = load_data()
demand = df["demand"]
yearly = demand.groupby(demand.index.year).sum() / 1e6
print(f"{len(df):,} hours, {df.index[0]:%Y-%m-%d} to {df.index[-1]:%Y-%m-%d}; demand {demand.min():,.0f}-{demand.max():,.0f} MW")
print("energy per year (TWh):", yearly.round(1).to_dict())
df.head(3)

26,304 hours, 2023-01-01 to 2025-12-31; demand 1,224-2,978 MW
energy per year (TWh): {2023: 17.6, 2024: 17.6, 2025: 17.9}


                     temperature  holiday       demand
2023-01-01 00:00:00     0.758841     True  1891.814698
2023-01-01 01:00:00     0.439160     True  1870.222676
2023-01-01 02:00:00     0.330122     True  1867.903308

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [2, 1]})
for label, start in [("a winter week", "2024-01-15"), ("a summer week", "2024-07-15")]:
    week = demand[start:pd.Timestamp(start) + pd.Timedelta(hours=167)]
    axes[0].plot(np.arange(168), week.to_numpy(), label=label)
axes[0].set(xlabel="hour of the week (Monday 00:00 = 0)", ylabel="MW", title="Daily and weekly rhythm")
axes[0].legend(fontsize=8)
daily = df.resample("D").agg({"temperature": "mean", "demand": "mean"})
axes[1].scatter(daily.temperature, daily.demand, s=4, alpha=0.4)
axes[1].set(xlabel="daily mean temperature (°C)", ylabel="daily mean demand (MW)", title="Heating and cooling")
plt.tight_layout()
plt.show()

Three years, 26,304 hours, 17.6–17.9 TWh a year — the size of Croatia's demand. The winter week sits higher all day (heating); the summer week has sharper midday peaks (cooling) and lower nights. The scatter shows why temperature matters: demand rises as days get colder *and* as they get hotter — a U shape with its minimum around 16–20 °C. It also shows **two bands**: workdays above, weekends and holidays below. Temperature and the type of day together explain most of the variation; the rest is weather and noise that persist from hour to hour. Every method in this module is, in the end, a way of capturing these patterns.

### Split in time

Train on 2023–2024, test on 2025. A random split — the default in many machine-learning tutorials — would put hours from 2025 into training and their neighbours into the test set; since demand an hour apart is almost the same, the model would be tested on data it has effectively seen. **In forecasting, the test set must lie in the future of everything used for training.**

In [4]:
train, test = demand[:"2024"], demand["2025"]
print(f"train {train.index[0]:%Y-%m-%d} to {train.index[-1]:%Y-%m-%d} ({len(train):,} h), "
      f"test {test.index[0]:%Y-%m-%d} to {test.index[-1]:%Y-%m-%d} ({len(test):,} h)")

train 2023-01-01 to 2024-12-31 (17,544 h), test 2025-01-01 to 2025-12-31 (8,760 h)


### What is known at noon?

A day-ahead forecast for day D is made at about **12:00 on day D−1** — before the day-ahead market closes. At that moment demand is known up to 11:00 on D−1. So the forecast for 00:00 on D is 13 hours ahead, the one for 23:00 on D is 36 hours ahead.

That rules out a tempting baseline. "Same hour yesterday" works for the morning hours of D (yesterday morning is known at noon) but **not for the afternoon**: when forecasting 15:00 on D, yesterday's 15:00 has not happened yet. A forecast that uses it anyway is using data from the future — a **leak**. The honest version falls back to the same hour two days earlier for every hour from 12:00 on:

In [5]:
def lag(series, hours, index):
    """Value `hours` earlier, aligned to `index`."""
    return series.shift(hours).loc[index]


leaky_yesterday = lag(demand, 24, test.index)                                       # uses the future!
yesterday = lag(demand, 24, test.index).where(test.index.hour < 12, lag(demand, 48, test.index))
last_week = lag(demand, 168, test.index)
hour_of_week = train.groupby([train.index.dayofweek, train.index.hour]).mean()        # climatology from training only
climatology = pd.Series(hour_of_week.loc[list(zip(test.index.dayofweek, test.index.hour))].to_numpy(), index=test.index)
assert (test.index.hour[yesterday.ne(leaky_yesterday)] >= 12).all(), "only the afternoon hours differ"
print("baselines ready: leaky yesterday, day-ahead yesterday (lag 24/48), last week (lag 168), climatology")

baselines ready: leaky yesterday, day-ahead yesterday (lag 24/48), last week (lag 168), climatology


### Metrics

With forecast f and actual y over n hours:

- **MAE** = mean |f − y| — in MW, easy to explain, the average miss
- **RMSE** = √mean (f − y)² — punishes large misses more; what matters if big errors are costly
- **bias** = mean (f − y) — systematic over- (+) or under-forecasting (−); errors that do not cancel out
- **MAPE** = mean |f − y| / y — in percent, comparable across systems, but meaningless when y is near zero
- **MASE** = MAE / MAE of the weekly naive forecast *on the training data* — below 1 beats that benchmark, above 1 does not; scale-free and robust

In [6]:
def scores(forecast, actual, scale):
    e = forecast - actual
    return {"MAE": e.abs().mean(), "RMSE": np.sqrt((e ** 2).mean()), "bias": e.mean(),
            "MAPE_%": 100 * (e.abs() / actual).mean(), "MASE": e.abs().mean() / scale}


mase_scale = (train - train.shift(168)).abs().mean()           # in-sample weekly naive, training data only
table = pd.DataFrame({
    "leaky: same hour yesterday": scores(leaky_yesterday, test, mase_scale),
    "same hour yesterday (lag 24/48)": scores(yesterday, test, mase_scale),
    "same hour last week (lag 168)": scores(last_week, test, mase_scale),
    "climatology (hour x weekday)": scores(climatology, test, mase_scale)}).T
print(f"MASE scale: weekly naive MAE on 2023-2024 = {mase_scale:.1f} MW")
table.round(2)

MASE scale: weekly naive MAE on 2023-2024 = 129.8 MW


                                    MAE    RMSE   bias  MAPE_%  MASE
leaky: same hour yesterday       138.37  177.27  -0.07    6.89  1.07
same hour yesterday (lag 24/48)  164.85  206.56  -0.20    8.16  1.27
same hour last week (lag 168)    142.50  182.46  -1.16    7.01  1.10
climatology (hour x weekday)     168.49  206.20 -29.04    8.22  1.30

Read the table from top to bottom.

**The leak flatters.** "Same hour yesterday" with data from the future scores an MAE of 138.4 MW — it would look like the best baseline. Done honestly, it scores **164.8 MW**, 19 % worse, and becomes one of the weakest. Leaks rarely look like bugs; they look like good results.

**Last week beats yesterday.** "Same hour last week" (142.5 MW) beats the honest "yesterday", because a one- or two-day lag crosses the weekly rhythm: Monday is forecast from Saturday or Sunday, Saturday from Thursday. A one-week lag always compares like with like.

**Bias reveals drift.** Climatology — the 2023–2024 average for each hour of the week — has a bias of −29 MW: demand in 2025 is a little higher (17.9 against 17.6 TWh), and an average of the past cannot follow that. The lag baselines have almost no bias because they always use recent data. MAE alone would not show the difference. (The exercise shows that this annual figure hides far larger seasonal biases that cancel over the year.)

**MASE** puts all four on one scale: every baseline scores above 1, i.e. worse on 2025 than the weekly naive forecast was on its own training years. Those numbers — about 140 MW, 7 % MAPE — are the bar the models of the next lessons must clear.

### Where do the forecasts fail?

In [7]:
holiday = df["holiday"].loc[test.index]
errors = {"yesterday (24/48)": (yesterday - test).abs(), "last week (168)": (last_week - test).abs(),
          "climatology": (climatology - test).abs()}
by_day_type = pd.DataFrame({name: {"normal days": e[~holiday].mean(), "holidays": e[holiday].mean()}
                            for name, e in errors.items()}).T
by_day_type["holiday_penalty_%"] = 100 * (by_day_type.holidays / by_day_type["normal days"] - 1)
by_day_type.round(1)

                   normal days  holidays  holiday_penalty_%
yesterday (24/48)        164.1     185.1               12.8
last week (168)          140.8     189.2               34.4
climatology              167.4     198.8               18.8

In [8]:
fig, ax = plt.subplots(figsize=(9, 3.4))
for name, e in errors.items():
    ax.plot(e.groupby(e.index.hour).mean(), marker="o", ms=3, label=name)
ax.axvline(11.5, color="grey", ls=":", lw=1)
ax.text(11.7, ax.get_ylim()[0] + 2, "from 12:00, 'yesterday' is two days old", fontsize=8, va="bottom")
ax.set(xlabel="hour of the forecast day", ylabel="MAE (MW)", title="Forecast error by hour, test year 2025")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

Every baseline does worse on **holidays**, and the weekly lag suffers most (+34 %): when last week's day was a holiday, or this week's is, copying last week copies the wrong kind of day. Holidays are rare but predictable — they are known years in advance — which makes them an easy gain for any model that is told about them.

By hour, the two lag baselines are equally good in the morning, about 125–135 MW. At **12:00 the honest "yesterday" jumps** to over 200 MW: from there on it must use data two days old. All methods do worse in the middle of the day, when demand depends most on that day's temperature — information none of the baselines uses. That is the opening for the next lesson: forecasts that use the weather.

### The professional benchmark: Croatia's TSO forecast

ENTSO-E publishes each TSO's day-ahead load forecast next to the actual load. With an ENTSO-E API key (M1.1) in `.env` as `ENTSOE_API_KEY`, the cell below scores HOPS's forecast for one month with the same metrics — the number your own models will have to beat in the next lessons. Without a key it is skipped.

In [9]:
def tso_forecast_scores(start, end, zone="HR"):
    """MAE, RMSE, bias and MAPE of the TSO's day-ahead load forecast (hourly), from ENTSO-E."""
    from entsoe import EntsoePandasClient
    client = EntsoePandasClient(api_key=os.environ["ENTSOE_API_KEY"])
    data = client.query_load_and_forecast(zone, start=pd.Timestamp(start, tz="Europe/Zagreb"),
                                          end=pd.Timestamp(end, tz="Europe/Zagreb"))
    hourly = data.resample("h").mean().dropna()
    e = hourly["Forecasted Load"] - hourly["Actual Load"]
    return {"hours": len(hourly), "MAE": e.abs().mean(), "RMSE": np.sqrt((e ** 2).mean()), "bias": e.mean(),
            "MAPE_%": 100 * (e.abs() / hourly["Actual Load"]).mean()}


try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
if os.environ.get("ENTSOE_API_KEY"):
    print(pd.Series(tso_forecast_scores("2025-01-01", "2025-02-01")).round(2))
else:
    print("No ENTSOE_API_KEY found - skipping the real-data comparison (see M1.1 for how to get a key).")

No ENTSOE_API_KEY found - skipping the real-data comparison (see M1.1 for how to get a key).


## Check your understanding

**Honest evaluation**

1. Why does "same hour yesterday" look *better* than "same hour last week" when it leaks, and worse when it does not? Which hours of the day does the leak affect?
2. A colleague evaluates a model with a random 80/20 split of all hours and reports an MAE of 40 MW. What would you ask before believing it?

**Metrics**

1. Climatology has a bias of about −29 MW in 2025. What does that tell you about 2025 compared with 2023–2024, and why does MAE alone not show it?
2. When would you report RMSE rather than MAE to a TSO? When would MAPE be misleading?